<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/05_Productivity_Classification_CHL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================
# STEP 31.9P-16R
# PRODUCTIVITY THRESHOLD DEFINITION
#
# PURPOSE:
# Define Low / Medium / High CHL productivity thresholds using ONLY
# the training CHL distribution.
#
# IMPORTANT METHODOLOGICAL RULES:
# - Training data ONLY
# - Validation data NOT loaded
# - Test data NOT loaded
# - No model retraining
# - No model modification
# - Thresholds are calculated before classification
#
# Productivity classes:
#   LOW    : CHL < 33rd percentile
#   MEDIUM : 33rd <= CHL <= 66th percentile
#   HIGH   : CHL > 66th percentile
# ======================================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive


print("=" * 70)
print("STEP 31.9P-16R")
print("PRODUCTIVITY THRESHOLD DEFINITION")
print("=" * 70)


# ======================================================================
# 1. MOUNT GOOGLE DRIVE
# ======================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ======================================================================
# 2. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESSING_DIR = (
    PROJECT_DIR / "05_model_preprocessing"
)

OUTPUT_DIR = (
    PROJECT_DIR /
    "06_models" /
    "18_productivity_classification"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_train.csv"
)

THRESHOLD_FILE = (
    OUTPUT_DIR /
    "productivity_thresholds_training_only.csv"
)

THRESHOLD_JSON = (
    OUTPUT_DIR /
    "productivity_thresholds_training_only.json"
)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("\nProject:")
print(PROJECT_DIR)

print("\nTraining dataset:")
print(TRAIN_FILE)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

print("\nTraining file exists:", TRAIN_FILE.exists())

if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Required training file not found: {TRAIN_FILE}"
    )

print("\n✓ Training dataset found.")


# ======================================================================
# 4. EXPLICIT DATA-SAFETY CHECK
# ======================================================================

VALIDATION_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_test.csv"
)

print("\n" + "=" * 70)
print("DATA USAGE SAFETY")
print("=" * 70)

print("""
TRAINING DATA : WILL BE USED
VALIDATION    : NOT LOADED
TEST DATA     : NOT LOADED

Thresholds will be calculated exclusively from the training
chlorophyll_a distribution.
""")

# Deliberately DO NOT load validation or test data.


# ======================================================================
# 5. LOAD TRAINING DATA
# ======================================================================

print("=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("\nTraining shape:")
print(train_df.shape)

print("\nTraining rows:", len(train_df))


# ======================================================================
# 6. CHECK TARGET COLUMN
# ======================================================================

TARGET_COLUMN = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET COLUMN CHECK")
print("=" * 70)

print("\nTarget column:", TARGET_COLUMN)

if TARGET_COLUMN not in train_df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' not found in training data."
    )

print("✓ chlorophyll_a column found.")


# ======================================================================
# 7. TARGET SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING CHL SAFETY CHECK")
print("=" * 70)

chl = pd.to_numeric(
    train_df[TARGET_COLUMN],
    errors="coerce"
)

nan_count = chl.isna().sum()
inf_count = np.isinf(chl.to_numpy()).sum()

print("\nCHL NaN values:", nan_count)
print("CHL infinite values:", inf_count)

if nan_count > 0:
    raise ValueError(
        "Training chlorophyll_a contains NaN values."
    )

if inf_count > 0:
    raise ValueError(
        "Training chlorophyll_a contains infinite values."
    )

if len(chl) == 0:
    raise ValueError(
        "No valid training CHL observations found."
    )

print("\n✓ Training CHL values passed safety checks.")


# ======================================================================
# 8. TRAINING CHL DISTRIBUTION
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING CHL DISTRIBUTION")
print("=" * 70)

print("\nNumber of CHL observations:", len(chl))

print(
    f"Minimum CHL       : {chl.min():.6f} mg/m³"
)

print(
    f"Maximum CHL       : {chl.max():.6f} mg/m³"
)

print(
    f"Mean CHL          : {chl.mean():.6f} mg/m³"
)

print(
    f"Median CHL        : {chl.median():.6f} mg/m³"
)


# ======================================================================
# 9. CALCULATE PRODUCTIVITY THRESHOLDS
#
# IMPORTANT:
# These percentiles are calculated ONLY from TRAINING CHL.
# ======================================================================

print("\n" + "=" * 70)
print("CALCULATING PRODUCTIVITY THRESHOLDS")
print("=" * 70)

LOW_MEDIUM_PERCENTILE = 33
MEDIUM_HIGH_PERCENTILE = 66

low_medium_threshold = np.percentile(
    chl,
    LOW_MEDIUM_PERCENTILE
)

medium_high_threshold = np.percentile(
    chl,
    MEDIUM_HIGH_PERCENTILE
)

print(
    f"\n33rd percentile threshold : "
    f"{low_medium_threshold:.6f} mg/m³"
)

print(
    f"66th percentile threshold : "
    f"{medium_high_threshold:.6f} mg/m³"
)


# ======================================================================
# 10. VERIFY THRESHOLD ORDER
# ======================================================================

print("\n" + "=" * 70)
print("THRESHOLD LOGIC CHECK")
print("=" * 70)

if not low_medium_threshold < medium_high_threshold:
    raise ValueError(
        "Invalid productivity thresholds: "
        "33rd percentile must be below 66th percentile."
    )

print(
    "\n✓ 33rd percentile < 66th percentile."
)

print(
    "✓ Three ordered productivity regions can be defined."
)


# ======================================================================
# 11. DEFINE PRODUCTIVITY CLASS RULES
# ======================================================================

print("\n" + "=" * 70)
print("PRODUCTIVITY CLASS DEFINITIONS")
print("=" * 70)

print("""
LOW:
    chlorophyll_a < 33rd percentile

MEDIUM:
    33rd percentile <= chlorophyll_a <= 66th percentile

HIGH:
    chlorophyll_a > 66th percentile
""")


# ======================================================================
# 12. APPLY CLASS RULES TO TRAINING DATA ONLY
#
# This is ONLY a diagnostic check to verify class balance.
# It does NOT involve validation or test data.
# ======================================================================

def classify_productivity(chl_value):
    if chl_value < low_medium_threshold:
        return "Low"
    elif chl_value <= medium_high_threshold:
        return "Medium"
    else:
        return "High"


training_classes = chl.apply(
    classify_productivity
)

class_counts = (
    training_classes
    .value_counts()
    .reindex(
        ["Low", "Medium", "High"],
        fill_value=0
    )
)

class_percentages = (
    training_classes
    .value_counts(normalize=True)
    .reindex(
        ["Low", "Medium", "High"],
        fill_value=0
    ) * 100
)


print("\n" + "=" * 70)
print("TRAINING PRODUCTIVITY CLASS DISTRIBUTION")
print("=" * 70)

distribution_df = pd.DataFrame({
    "productivity_class": class_counts.index,
    "observations": class_counts.values,
    "percentage": class_percentages.values
})

print(
    distribution_df.to_string(
        index=False
    )
)


# ======================================================================
# 13. SAVE THRESHOLDS
# ======================================================================

print("\n" + "=" * 70)
print("SAVING PRODUCTIVITY THRESHOLDS")
print("=" * 70)

threshold_df = pd.DataFrame({
    "threshold_type": [
        "low_medium_boundary",
        "medium_high_boundary"
    ],
    "percentile": [
        LOW_MEDIUM_PERCENTILE,
        MEDIUM_HIGH_PERCENTILE
    ],
    "threshold_mg_m3": [
        low_medium_threshold,
        medium_high_threshold
    ],
    "source_dataset": [
        "training_only",
        "training_only"
    ],
    "target_variable": [
        TARGET_COLUMN,
        TARGET_COLUMN
    ]
})

threshold_df.to_csv(
    THRESHOLD_FILE,
    index=False
)

print("\nSaved:")
print(THRESHOLD_FILE)


# ======================================================================
# 14. SAVE JSON CONFIGURATION
# ======================================================================

threshold_config = {
    "method": "training_distribution_percentile",
    "target_variable": TARGET_COLUMN,
    "source_dataset": "final_model_ready_train.csv",
    "validation_data_used": False,
    "test_data_used": False,
    "low_medium_percentile": LOW_MEDIUM_PERCENTILE,
    "medium_high_percentile": MEDIUM_HIGH_PERCENTILE,
    "low_medium_threshold_mg_m3": float(
        low_medium_threshold
    ),
    "medium_high_threshold_mg_m3": float(
        medium_high_threshold
    ),
    "classification_rules": {
        "Low": (
            f"CHL < {low_medium_threshold:.10f}"
        ),
        "Medium": (
            f"{low_medium_threshold:.10f} <= CHL "
            f"<= {medium_high_threshold:.10f}"
        ),
        "High": (
            f"CHL > {medium_high_threshold:.10f}"
        )
    }
}

with open(
    THRESHOLD_JSON,
    "w"
) as f:
    json.dump(
        threshold_config,
        f,
        indent=4
    )

print("\nSaved:")
print(THRESHOLD_JSON)


# ======================================================================
# 15. FINAL SAFETY VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16R THRESHOLD SAFETY CHECK")
print("=" * 70)

print(
    "\nTraining data used        : TRUE"
)

print(
    "Validation data loaded    : FALSE"
)

print(
    "Test data loaded          : FALSE"
)

print(
    "Test data used            : FALSE"
)

print(
    "Model retraining          : FALSE"
)

print(
    "Model modification        : FALSE"
)

print(
    "Threshold source          : TRAINING ONLY"
)

print(
    f"33rd percentile           : "
    f"{low_medium_threshold:.6f} mg/m³"
)

print(
    f"66th percentile           : "
    f"{medium_high_threshold:.6f} mg/m³"
)

print("\n✓ Thresholds derived exclusively from training data.")
print("✓ No validation information used.")
print("✓ No test information used.")
print("✓ No model was retrained.")
print("✓ No existing model was modified.")
print("✓ Productivity thresholds saved successfully.")


# ======================================================================
# 16. COMPLETE
# ======================================================================

print("\n" + "=" * 70)
print("STEP 31.9P-16R — THRESHOLD DEFINITION COMPLETE")
print("=" * 70)

print("""
✓ Training CHL distribution analysed.
✓ 33rd percentile calculated.
✓ 66th percentile calculated.
✓ Low / Medium / High boundaries defined.
✓ Training class distribution verified.
✓ Threshold CSV saved.
✓ Threshold JSON saved.
✓ Validation data NOT loaded.
✓ Test data NOT loaded.
✓ Test data NOT used.
""")

STEP 31.9P-16R
PRODUCTIVITY THRESHOLD DEFINITION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Training dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Output directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/18_productivity_classification

REQUIRED FILE CHECK

Training file exists: True

✓ Training dataset found.

DATA USAGE SAFETY

TRAINING DATA : WILL BE USED
VALIDATION    : NOT LOADED
TEST DATA     : NOT LOADED

Thresholds will be calculated exclusively from the training
chlorophyll_a distribution.

LOADING TRAINING DATA

Training shape:
(736950, 75)

Training rows: 736950

TARGET COLUMN CHECK

Target column: chlorophyll_a
✓ chlorophyll_a column found.

TRAINING CHL SAFETY CHECK

CHL NaN values: 0
CHL infinite values: 0

✓ Training CHL values 